# 03 — First-move pilot: preparation, collection and audit

This is a new outcome and experiment version. Earlier full-sequence scores remain unchanged. The fixed assisted prompt requests only the official first solver move; full-sequence completion is not inferred from it.

The pilot uses 48 fresh puzzles, with 8 in each rating × solution-length cell (24 one-move, 24 multi-move). This deliberately balanced sample does not estimate natural Lichess accuracy. All 150 original or previously reserved IDs are excluded; the reserved 75 remain unused.

Run all cells with the project `.venv` kernel and Ollama running. Collection is resumable. Each request and outcome is saved immediately. The settings and executable cells are frozen before the first request; do not edit them after collection begins. Results analysis is in notebook 04.

In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib, json, re, time, uuid, zipfile
import numpy as np
import pandas as pd
import chess
import requests
from IPython.display import display

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'experiments/first_move_v1/config.json').is_file())
CONFIG=json.loads((ROOT/'experiments/first_move_v1/config.json').read_text())
PROMPT=(ROOT/'experiments/first_move_v1/prompt.txt').read_text()
OUT=ROOT/'data/experiments/first_move_v1';OUT.mkdir(parents=True,exist_ok=True)
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def save(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2),encoding='utf-8')
def records(name):
    p=OUT/name
    return [json.loads(s) for s in p.read_text(encoding='utf-8').splitlines() if s.strip()] if p.exists() else []
def append(name,value):
    with (OUT/name).open('a',encoding='utf-8') as stream:
        stream.write(json.dumps(value)+'\n');stream.flush()
def historical_integrity():
    expected=json.loads((ROOT/'archive/code_snapshot_hashes.json').read_text())
    with zipfile.ZipFile(ROOT/'archive/pre_first_move_code.zip') as z:
        assert set(z.namelist())==set(expected)
        assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in expected.items())
        protected=json.loads((ROOT/'data/experiments/board_prompt_v1/checkpoint_hashes.json').read_text())
        for name,value in protected.items():
            actual=hashlib.sha256(z.read(name)).hexdigest() if name.startswith('code/') else sha(ROOT/name)
            assert actual==value,name
    return len(expected),len(protected)
print('Verified archive and original checkpoint:',historical_integrity())


Verified archive and original checkpoint: (22, 23)


## 1. Review the five previously correct assisted first moves

Verify exact agreement with the official answer, legality, and the subsequent failed response. This validates scoring, not optimality or uniqueness of the Lichess line. Incorrect legal alternatives are retained for later engine adjudication if needed.

In [2]:
old=ROOT/'data/experiments/board_prompt_v1'
old_outcomes=pd.DataFrame([json.loads(s) for s in (old/'outcomes.jsonl').read_text().splitlines()])
old_outcomes=old_outcomes.drop_duplicates(['stage','condition','llm_name','puzzle_id'],keep='last')
correct=old_outcomes.loc[old_outcomes.stage.eq('development') & old_outcomes.condition.eq('legal_choices_json') & old_outcomes.correct_solver_moves.ge(1)]
assert len(correct)==5
features=pd.read_csv(old/'development_puzzles.csv',dtype={'puzzle_id':str}).set_index('puzzle_id')
logs=defaultdict(list)
for line in (old/'requests.jsonl').read_text().splitlines():
    entry=json.loads(line)
    if 'response' in entry:logs[entry['trial_id']].append(entry)
review=[]
for row in correct.itertuples(index=False):
    puzzle=features.loc[row.puzzle_id];board=chess.Board(puzzle.solver_fen)
    calls=sorted(logs[row.trial_id],key=lambda r:r['solver_turn'])
    move=json.loads(calls[0]['response']['message']['content'])['move']
    expected=puzzle.official_moves.split()[1]
    assert move==expected and chess.Move.from_uci(move) in board.legal_moves
    assert puzzle.solver_fen in calls[0]['request']['messages'][0]['content']
    review.append({'model':row.llm_name,'puzzle_id':row.puzzle_id,'returned_first_move':move,
        'official_first_move':expected,'legal_and_matches':True,'later_failure':row.failure_type,
        'later_response':calls[-1]['response']['message']['content'],'game_url':puzzle.game_url})
pd.DataFrame(review).to_csv(OUT/'prior_five_first_moves_review.csv',index=False)
display(pd.DataFrame(review))


,model,puzzle_id,returned_first_move,official_first_move,legal_and_matches,later_failure,later_response,game_url
0,gemma3:4b,1Z6xa,g7h6,g7h6,True,legal_incorrect,"{""move"":""f7f6""}",https://lichess.org/d6ZY0NQk#79
1,gemma3:4b,YDjpU,d4e2,d4e2,True,legal_incorrect,"{""move"":""d5c7""}",https://lichess.org/ECUrLa2j/black#30
2,qwen3:8b,1Z6xa,g7h6,g7h6,True,legal_incorrect,"{""move"":""f2f5""}",https://lichess.org/d6ZY0NQk#79
3,qwen3:8b,VSuYG,e3f5,e3f5,True,legal_incorrect,"{""move"":""f5d4""}",https://lichess.org/V0RTH1lO#109
4,llama3.1:8b,eaXbo,e3d4,e3d4,True,legal_incorrect,"{""move"":""e8c6""}",https://lichess.org/Ij4IvBFR#109


## 2. Select fresh puzzles and freeze the train/test split

No rating-quality filter is added. Existing human-rating boundaries are retained (≤1157; 1158–1700; >1700). Within each rating group, sample 8 one-move and 8 multi-move puzzles uniformly by random priorities. Solution length is a sampling dimension and predictor, not a prompt hint.

The 48 IDs are selected before responses are collected. Six per cell go to training and two to testing; all three LLM attempts on a puzzle stay together.

In [3]:
sample_path=OUT/'puzzles.csv'
if sample_path.exists():
    sample=pd.read_csv(sample_path,dtype={'puzzle_id':str})
    preparation=json.loads((OUT/'preparation.json').read_text())
    assert sha(sample_path)==preparation['sample_sha256']
else:
    excluded=set(pd.read_csv(ROOT/'data/samples/pilot_v1/pilot_puzzles.csv',dtype={'puzzle_id':str}).puzzle_id)
    excluded.update(pd.read_csv(ROOT/'data/experiments/board_prompt_v1/evaluation_puzzles.csv',dtype={'puzzle_id':str}).puzzle_id)
    assert len(excluded)==150
    rng=np.random.default_rng(CONFIG['seed'])
    candidates={(rating,length):pd.DataFrame() for rating in ['low','middle','high'] for length in ['one_move','multi_move']}
    population=defaultdict(int)
    for chunk in pd.read_csv(ROOT/'data/validation/full/validated_features.csv',chunksize=100_000,dtype={'puzzle_id':str}):
        frame=chunk.loc[~chunk.puzzle_id.isin(excluded)].copy()
        lower,upper=CONFIG['rating_cutoffs']
        frame['rating_stratum']=np.where(frame.puzzle_rating<=lower,'low',np.where(frame.puzzle_rating<=upper,'middle','high'))
        frame['length_group']=np.where(frame.total_solver_moves.eq(1),'one_move','multi_move')
        frame['_priority']=rng.random(len(frame))
        for key in candidates:
            group=frame.loc[frame.rating_stratum.eq(key[0]) & frame.length_group.eq(key[1])]
            population['/'.join(key)]+=len(group)
            candidates[key]=pd.concat([candidates[key],group.nsmallest(8,'_priority')],ignore_index=True).nsmallest(8,'_priority')
    sample=pd.concat(candidates.values(),ignore_index=True).drop(columns='_priority')
    wanted=set(sample.puzzle_id);assert len(wanted)==48 and not wanted & excluded
    source=ROOT/'data/lichess_db_puzzle.csv.zst'
    expected_hash=json.loads((ROOT/'data/validation/full/summary.json').read_text())['sha256']
    assert sha(source)==expected_hash
    found=[]
    for chunk in pd.read_csv(source,compression='zstd',chunksize=100_000,dtype=str,keep_default_na=False,
                              usecols=['PuzzleId','FEN','Moves','GameUrl']):
        matches=chunk.loc[chunk.PuzzleId.isin(wanted)]
        if not matches.empty:found.append(matches)
    answer=pd.concat(found).rename(columns={'PuzzleId':'puzzle_id','FEN':'original_fen','Moves':'official_moves','GameUrl':'game_url'})
    sample=sample.merge(answer,on='puzzle_id',validate='one_to_one')
    for row in sample.itertuples(index=False):
        board=chess.Board(row.original_fen);assert board.is_valid()
        for i,text in enumerate(row.official_moves.split()):
            assert chess.Move.from_uci(text) in board.legal_moves;board.push_uci(text)
            if i==0:
                assert board.fen()==row.solver_fen and board.legal_moves.count()==row.legal_moves
        assert len(row.official_moves.split()[1::2])==row.total_solver_moves
    sample=sample.sort_values('puzzle_id').reset_index(drop=True)
    train_ids=set(pd.concat([group.sample(6,random_state=CONFIG['seed']) for _,group in sample.groupby(['rating_stratum','length_group'])]).puzzle_id)
    sample['partition']=np.where(sample.puzzle_id.isin(train_ids),'train','test')
    sample['official_first_move']=sample.official_moves.str.split().str[1]
    sample['random_success_probability']=1/sample.legal_moves
    sample.to_csv(sample_path,index=False)
    preparation={'seed':CONFIG['seed'],'n':48,'excluded_ids':sorted(excluded),'source_sha256':expected_hash,
        'population_cell_counts':dict(population),'sample_sha256':sha(sample_path),
        'note':'Balanced diagnostic sample, not a population accuracy estimate; 36 train and 12 test puzzles fixed before inference.'}
    save('preparation.json',preparation)
assert sample.puzzle_id.is_unique and len(sample)==48
assert sample.groupby(['rating_stratum','length_group']).size().eq(8).all()
assert sample.partition.value_counts().to_dict()=={'train':36,'test':12}
assert not set(sample.puzzle_id)&set(preparation['excluded_ids'])
display(sample.groupby(['rating_stratum','length_group','partition']).size().rename('n').to_frame())
print('Exact expected uniform-random correct moves per model:',sample.random_success_probability.sum())


n
rating_stratum length_group partition   
high           multi_move   test       2
                            train      6
               one_move     test       2
                            train      6
low            multi_move   test       2
                            train      6
               one_move     test       2
                            train      6
middle         multi_move   test       2
                            train      6
               one_move     test       2
                            train      6

Exact expected uniform-random correct moves per model: 1.983844951840387


## 3. Fixed request, scoring and checks

Every model receives the same current position, piece list and alphabetically sorted legal UCI choices. JSON output is constrained to those choices. This assists legality; it does not supply the correct answer or engine scores. Qwen thinking is disabled and the output budget is fixed at 256 tokens.

Success is exact agreement with the official first move. Generation-limit and format failures count as protocol failures. Infrastructure failures are retried once and otherwise left unscored. This pilot measures assisted first-move selection, not full puzzle solving.

In [4]:
def request_body(row,model):
    board=chess.Board(row.solver_fen)
    pieces=[]
    for color in [chess.WHITE,chess.BLACK]:
        entries=[f'{chess.piece_name(piece.piece_type)} {chess.square_name(square)}' for square,piece in sorted(board.piece_map().items()) if piece.color==color]
        pieces.append(('White' if color else 'Black')+' pieces: '+', '.join(entries))
    choices=sorted(move.uci() for move in board.legal_moves)
    prompt=PROMPT.format(side='White' if board.turn else 'Black',fen=board.fen(),pieces='\n'.join(pieces),choices=', '.join(choices))
    body={'model':model,'messages':[{'role':'user','content':prompt}],'stream':False,'keep_alive':'5m','options':CONFIG['options'],
        'format':{'type':'object','properties':{'move':{'type':'string','enum':choices}},'required':['move'],'additionalProperties':False}}
    if model=='qwen3:8b':body['think']=CONFIG['qwen_think']
    return body
def score(row,payload):
    if payload.get('done_reason')=='length':return 'generation_limit',None
    try:
        value=json.loads(payload['message']['content'])
        assert isinstance(value,dict) and set(value)=={'move'}
        move=value['move'];assert isinstance(move,str) and re.fullmatch('[a-h][1-8][a-h][1-8][qrbn]?',move)
    except (ValueError,KeyError,AssertionError,TypeError):return 'unparseable',None
    board=chess.Board(row.solver_fen)
    if chess.Move.from_uci(move) not in board.legal_moves:return 'illegal_move',move
    return ('success' if move==row.official_first_move else 'legal_incorrect'),move
from types import SimpleNamespace
fixture=SimpleNamespace(solver_fen=chess.STARTING_FEN,official_first_move='e2e4')
def fake(text,reason='stop'):return {'done_reason':reason,'message':{'content':text}}
assert score(fixture,fake('{"move":"e2e4"}'))[0]=='success'
assert score(fixture,fake('{"move":"d2d4"}'))[0]=='legal_incorrect'
assert score(fixture,fake('{"move":"e2e5"}'))[0]=='illegal_move'
assert score(fixture,fake('e2e4'))[0]=='unparseable'
assert score(fixture,fake('{"move":"e2e4"}','length'))[0]=='generation_limit'
assert set(request_body(fixture,'qwen3:8b')['format']['properties']['move']['enum'])=={m.uci() for m in chess.Board().legal_moves}
print('First-move scoring and request checks passed.')


First-move scoring and request checks passed.


## 4. Freeze provenance and collect 144 attempts

This cell makes local Ollama inference calls. It skips completed puzzle/model pairs on rerun. A changed model digest, sample, configuration, prompt, or executable notebook cell is rejected; use another experiment version for changes.

In [5]:
tags_response=requests.get(CONFIG['base_url']+'/api/tags',timeout=10);tags_response.raise_for_status()
tags={m['name']:m for m in tags_response.json()['models']}
models={}
for name in CONFIG['models']:
    assert name in tags,'Missing model: '+name
    response=requests.post(CONFIG['base_url']+'/api/show',json={'model':name},timeout=15);response.raise_for_status()
    details=response.json()
    models[name]={'digest':tags[name]['digest'],'details':details.get('details'),'capabilities':details.get('capabilities')}
import nbformat
notebook=nbformat.read(ROOT/'notebooks/03_first_move_pilot.ipynb',as_version=4)
code_hash=hashlib.sha256('\n\n'.join(c.source for c in notebook.cells if c.cell_type=='code').encode()).hexdigest()
manifest={'config':CONFIG,'prompt':PROMPT,'sample_sha256':sha(sample_path),'notebook_code_sha256':code_hash,
          'models':models,'archive_sha256':sha(ROOT/'archive/pre_first_move_code.zip')}
if (OUT/'manifest.json').exists():assert json.loads((OUT/'manifest.json').read_text())==manifest,'Frozen experiment changed'
else:save('manifest.json',manifest)
completed={(r['puzzle_id'],r['llm_name']) for r in records('outcomes.jsonl') if r['status']=='completed'}
for model in CONFIG['models']:
    for row in sample.itertuples(index=False):
        if (row.puzzle_id,model) in completed:continue
        trial_id=str(uuid.uuid4());body=request_body(row,model);started=time.perf_counter();payload=None
        for retry in range(2):
            log={'trial_id':trial_id,'puzzle_id':row.puzzle_id,'llm_name':model,'retry':retry,
                 'timestamp_utc':datetime.now(timezone.utc).isoformat(),'request':body}
            try:
                response=requests.post(CONFIG['base_url']+'/api/chat',json=body,timeout=(10,180));response.raise_for_status()
                value=response.json()
                if not value.get('done') or 'content' not in value.get('message',{}):raise ValueError('Incomplete response')
                log['response']=value;payload=value
            except (requests.RequestException,ValueError) as error:
                log['error_type']=type(error).__name__
                log['http_status']=getattr(getattr(error,'response',None),'status_code',None)
            append('requests.jsonl',log)
            if payload is not None:break
        category,move=score(row,payload) if payload is not None else ('request_failed',None)
        result={'trial_id':trial_id,'puzzle_id':row.puzzle_id,'llm_name':model,
            'status':'completed' if payload is not None else 'infrastructure_error',
            'success':int(category=='success') if payload is not None else None,'failure_type':category,
            'returned_move':move,'elapsed_seconds':round(time.perf_counter()-started,3)}
        append('outcomes.jsonl',result)
        print(model,row.puzzle_id,category,flush=True)
    response=requests.post(CONFIG['base_url']+'/api/generate',json={'model':model,'keep_alive':0},timeout=60);response.raise_for_status()
outcomes=pd.DataFrame(records('outcomes.jsonl')).drop_duplicates(['puzzle_id','llm_name'],keep='last')
outcomes.to_csv(OUT/'outcomes.csv',index=False)
display(outcomes.groupby('llm_name').agg(n=('puzzle_id','size'),solved=('success','sum'),mean_seconds=('elapsed_seconds','mean')))


gemma3:4b 23Azo legal_incorrect


gemma3:4b 3ZL1k legal_incorrect


gemma3:4b 3uqwM legal_incorrect


gemma3:4b 4LBH5 legal_incorrect


gemma3:4b 4hd0x legal_incorrect


...

llama3.1:8b uthr0 legal_incorrect


llama3.1:8b v1Sb0 legal_incorrect


llama3.1:8b vFqWb legal_incorrect


llama3.1:8b y8mcw legal_incorrect


,n,solved,mean_seconds
llm_name,,,
gemma3:4b,48,1,1.574458
llama3.1:8b,48,2,1.897417
qwen3:8b,48,3,1.950458


## 5. Independently audit recorded attempts and historical preservation

Rebuild each expected request and score the saved response without trusting the saved success label. Verify all models attempted the same 48 puzzles and that earlier checkpoints still match their recorded hashes.

In [6]:
requests_by_trial=defaultdict(list)
for log in records('requests.jsonl'):
    if 'response' in log:requests_by_trial[log['trial_id']].append(log)
indexed=sample.set_index('puzzle_id')
assert len(outcomes)==144 and outcomes.groupby('llm_name').size().eq(48).all()
for result in outcomes.itertuples(index=False):
    row=SimpleNamespace(**indexed.loc[result.puzzle_id].to_dict())
    logs=requests_by_trial[result.trial_id]
    if result.status=='completed':
        assert len(logs)==1
        assert logs[0]['request']==request_body(row,result.llm_name)
        category,move=score(row,logs[0]['response'])
        assert result.success==int(category=='success') and result.failure_type==category and result.returned_move==move
    else:assert pd.isna(result.success)
archive_n,protected_n=historical_integrity()
save('audit_summary.json',{'attempts_verified':144,'archive_files_verified':archive_n,
    'original_checkpoint_files_verified':protected_n,'prior_first_moves_verified':5,
    'infrastructure_errors':int(outcomes.status.ne('completed').sum()),'status':'passed'})
print('Audit passed. Open 04_first_move_results.ipynb for chance comparisons and preliminary models.')


Audit passed. Open 04_first_move_results.ipynb for chance comparisons and preliminary models.
